# Week 25, Day 3: Project: files and persistence

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w025d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('money.py', 'w', encoding='utf-8') as f:
    f.write('"""Reading and showing money and categories."""\nimport re\n\nAMOUNT = re.compile(r"£?\\s*(\\d{1,3}(?:,\\d{3})+|\\d+)(?:\\.(\\d{1,2}))?")\n\n\ndef parse_amount(text):\n    """Text such as \'£1,250.50\' as whole pence. Raises ValueError with a readable message."""\n    text = text.strip()\n    if not text.lstrip("£").strip():\n        raise ValueError("missing amount")\n    m = AMOUNT.fullmatch(text)\n    if m is None:\n        raise ValueError(f"not an amount: {text!r}")\n    total = int(m.group(1).replace(",", "")) * 100 + int((m.group(2) or "").ljust(2, "0"))\n    if total == 0:\n        raise ValueError("amount must be more than £0")\n    return total\n\n\ndef format_pence(pence):\n    """1250 as \'£12.50\', -350 as \'-£3.50\'."""\n    sign = "-" if pence < 0 else ""\n    pounds, rest = divmod(abs(pence), 100)\n    return f"{sign}£{pounds:,}.{rest:02d}"\n\n\ndef clean_category(text):\n    """\' Eating   Out \' as \'eating out\'. Raises ValueError if it isn\'t words of letters."""\n    shown = text.strip()\n    if not shown:\n        raise ValueError("missing category")\n    cleaned = " ".join(shown.lower().split())\n    if len(cleaned) > 20 or not re.fullmatch(r"[a-z]+( [a-z]+)*", cleaned):\n        raise ValueError(f"bad category: {shown!r}")\n    return cleaned\n')
with open('dates.py', 'w', encoding='utf-8') as f:
    f.write('"""Reading dates and months that people type, and stepping through months."""\nimport re\nfrom datetime import datetime, timedelta\n\nFORMATS = ["%Y-%m-%d", "%d/%m/%Y"]\n\n\ndef parse_date(text, today):\n    """\'2026-11-02\', \'02/11/2026\', \'today\', \'yesterday\' or \'\' as ISO text. No future dates."""\n    shown = text.strip()\n    word = shown.lower()\n    if word in ("", "today"):\n        when = today\n    elif word == "yesterday":\n        when = today - timedelta(days=1)\n    else:\n        when = None\n        for fmt in FORMATS:\n            try:\n                when = datetime.strptime(word, fmt).date()\n                break\n            except ValueError:\n                pass\n        if when is None:\n            raise ValueError(f"not a date: {shown!r}")\n    if when > today:\n        raise ValueError(f"date is in the future: {when.isoformat()}")\n    return when.isoformat()\n\n\ndef parse_month(text):\n    """\'2026-11\' or \'11/2026\' as \'2026-11\'. Raises ValueError otherwise."""\n    shown = text.strip()\n    m = re.fullmatch(r"(\\d{4})-(\\d{2})", shown)\n    if m:\n        year, month = int(m.group(1)), int(m.group(2))\n    else:\n        m = re.fullmatch(r"(\\d{1,2})/(\\d{4})", shown)\n        if not m:\n            raise ValueError(f"not a month: {shown!r}")\n        month, year = int(m.group(1)), int(m.group(2))\n    if not 1 <= month <= 12:\n        raise ValueError(f"not a month: {shown!r}")\n    return f"{year}-{month:02d}"\n\n\ndef next_month(month):\n    """\'2026-12\' gives \'2027-01\'."""\n    year, m = int(month[:4]), int(month[5:])\n    if m == 12:\n        return f"{year + 1}-01"\n    return f"{year}-{m + 1:02d}"\n')
with open('core.py', 'w', encoding='utf-8') as f:
    f.write('"""The expense tracker\'s data and the operations on it."""\nfrom money import parse_amount, clean_category\nfrom dates import parse_date, next_month\n\n\ndef new_data():\n    return {"expenses": [], "budgets": {}, "next_id": 1}\n\n\ndef add_expense(data, amount_text, category_text, note, date_text, today):\n    """Check every field, then add the expense, or raise one ValueError listing every problem."""\n    problems = []\n    amount = category = when = None\n    try:\n        amount = parse_amount(amount_text)\n    except ValueError as e:\n        problems.append(str(e))\n    try:\n        category = clean_category(category_text)\n    except ValueError as e:\n        problems.append(str(e))\n    try:\n        when = parse_date(date_text, today)\n    except ValueError as e:\n        problems.append(str(e))\n    note = note.strip()\n    if len(note) > 60:\n        problems.append("note is longer than 60 characters")\n    if problems:\n        raise ValueError("; ".join(problems))\n    expense = {"id": data["next_id"], "date": when, "amount": amount, "category": category, "note": note}\n    data["expenses"].append(expense)\n    data["next_id"] += 1\n    return expense\n\n\ndef remove_expense(data, expense_id):\n    """Remove and return the expense with this id. KeyError if there isn\'t one."""\n    for i, e in enumerate(data["expenses"]):\n        if e["id"] == expense_id:\n            return data["expenses"].pop(i)\n    raise KeyError(f"no expense #{expense_id}")\n\n\ndef filter_expenses(data, month=None, category=None):\n    """A new list of the matching expenses, newest first (by date, then id)."""\n    chosen = [e for e in data["expenses"]\n              if (month is None or e["date"][:7] == month)\n              and (category is None or e["category"] == category)]\n    return sorted(chosen, key=lambda e: (e["date"], e["id"]), reverse=True)\n\n\ndef total(expenses):\n    return sum(e["amount"] for e in expenses)\n\n\ndef by_category(expenses):\n    """(category, pence) pairs, largest first, alphabetical for ties."""\n    totals = {}\n    for e in expenses:\n        totals[e["category"]] = totals.get(e["category"], 0) + e["amount"]\n    return sorted(totals.items(), key=lambda pair: (-pair[1], pair[0]))\n\n\ndef month_summary(data, month):\n    """Count, total, a row per category with spending or a budget, and the categories over budget."""\n    expenses = filter_expenses(data, month=month)\n    spent = dict(by_category(expenses))\n    rows = []\n    for category in set(spent) | set(data["budgets"]):\n        budget = data["budgets"].get(category)\n        used = spent.get(category, 0)\n        rows.append({"category": category, "spent": used, "budget": budget,\n                     "left": None if budget is None else budget - used})\n    rows.sort(key=lambda r: (-r["spent"], r["category"]))\n    return {"month": month, "count": len(expenses), "total": total(expenses), "rows": rows,\n            "over": [r["category"] for r in rows if r["left"] is not None and r["left"] < 0]}\n\n\ndef monthly_totals(data):\n    """(month, pence) for every month from the first with spending to the last, gaps as 0."""\n    totals = {}\n    for e in data["expenses"]:\n        totals[e["date"][:7]] = totals.get(e["date"][:7], 0) + e["amount"]\n    if not totals:\n        return []\n    out = []\n    month, last = min(totals), max(totals)\n    while month <= last:\n        out.append((month, totals.get(month, 0)))\n        month = next_month(month)\n    return out\n\n\ndef set_budget(data, category_text, amount_text):\n    """Check both, then set the category\'s monthly budget. Returns (category, pence)."""\n    problems = []\n    category = amount = None\n    try:\n        category = clean_category(category_text)\n    except ValueError as e:\n        problems.append(str(e))\n    try:\n        amount = parse_amount(amount_text)\n    except ValueError as e:\n        problems.append(str(e))\n    if problems:\n        raise ValueError("; ".join(problems))\n    data["budgets"][category] = amount\n    return category, amount\n\n\ndef remove_budget(data, category_text):\n    """Remove a category\'s budget and return it. KeyError if it has none."""\n    category = clean_category(category_text)\n    if category not in data["budgets"]:\n        raise KeyError(f"no budget for {category!r}")\n    return data["budgets"].pop(category)\n')
with open('statement.csv', 'w', encoding='utf-8') as f:
    f.write('Date,Description,Amount,Balance\n02/11/2026,TESCO STORES 2041,-62.10,1702.15\n02/11/2026,TFL TRAVEL CH,-4.20,1764.25\n02/11/2026,TFL TRAVEL CH,-4.20,1768.45\n01/11/2026,SALARY,"2,500.00",1772.65\n01/11/2026,CITY LETTINGS RENT,"-3,200.00",-727.35\n31/10/2026,NETFLIX.COM,-10.99,2472.65\n31/10/2026,CORNER CAFE,-3.80,2483.64\n30/10/2026,  PRET   A MANGER ,-6.45,2487.44\n3O/10/2026,UBER *TRIP,-14.00,2493.89\n29/10/2026,AMAZON MKTPLACE,-1O.00,2507.89\n')

### Exercise: Step 14: load the data

Write `load_data(path)`:

1. if there's no file at `path`, return `new_data()`;
2. if the file isn't valid JSON, raise `ValueError(f"{path} is damaged: {e.msg} at line {e.lineno}")`, where `e` is the `json.JSONDecodeError` (end the `raise` with `from None`, so the message isn't buried under the first error);
3. if it isn't a dictionary with an `"expenses"` list, a `"budgets"` dictionary and an `int` `"next_id"`, raise `ValueError(f"{path} is not a spend data file")`;
4. if any expense isn't a dictionary with exactly the keys `id, date, amount, category, note` in that order and an `int` amount, raise `ValueError(f"{path} has a damaged expense: {e!r}")` with that expense;
5. otherwise return the data.

In [ ]:
import json
import os
from core import new_data

KEYS = ["id", "date", "amount", "category", "note"]

def load_data(path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import json as _json\nfrom core import new_data as _new\ndef message(path):\n    try:\n        load_data(path)\n    except ValueError as e:\n        return str(e)\n    raise AssertionError(f\"load_data({path!r}) should raise ValueError\")\ndef test_missing():\n    \"no file gives new data\"\n    assert load_data(\"nothing.json\") == _new()\ndef test_good():\n    \"a good file\"\n    d = {\"expenses\": [{\"id\": 1, \"date\": \"2026-11-02\", \"amount\": 1250, \"category\": \"food\", \"note\": \"lunch\"}],\n         \"budgets\": {\"food\": 6000}, \"next_id\": 2}\n    write_file(\"spend.json\", _json.dumps(d))\n    assert load_data(\"spend.json\") == d\ndef test_damaged():\n    \"damaged JSON\"\n    write_file(\"spend.json\", '{\"expenses\": [\\n  {\"id\": 1,\\n')\n    assert message(\"spend.json\") == \"spend.json is damaged: Expecting property name enclosed in double quotes at line 3\"\n    write_file(\"empty.json\", \"\")\n    assert message(\"empty.json\") == \"empty.json is damaged: Expecting value at line 1\"\ndef test_wrong():\n    \"not a spend file\"\n    for text in (\"[1, 2]\", '{\"expenses\": [], \"budgets\": {}}', '{\"expenses\": {}, \"budgets\": {}, \"next_id\": 1}',\n                 '{\"expenses\": [], \"budgets\": [], \"next_id\": 1}', '{\"expenses\": [], \"budgets\": {}, \"next_id\": \"1\"}'):\n        write_file(\"other.json\", text)\n        assert message(\"other.json\") == \"other.json is not a spend data file\", text\ndef test_bad_expense():\n    \"a damaged expense\"\n    bad = {\"id\": 1, \"date\": \"2026-11-02\", \"amount\": 12.5, \"category\": \"food\", \"note\": \"\"}\n    write_file(\"spend.json\", _json.dumps({\"expenses\": [bad], \"budgets\": {}, \"next_id\": 2}))\n    assert message(\"spend.json\") == f\"spend.json has a damaged expense: {bad!r}\"\n    bad2 = {\"id\": 1, \"date\": \"2026-11-02\", \"amount\": 1250, \"category\": \"food\"}\n    write_file(\"spend.json\", _json.dumps({\"expenses\": [bad2], \"budgets\": {}, \"next_id\": 2}))\n    assert message(\"spend.json\") == f\"spend.json has a damaged expense: {bad2!r}\"")

### Exercise: Step 15: save atomically

Write `save_data(path, data)` that saves `data` as JSON (with `indent=2` and `ensure_ascii=False`, so a note like `"Zoë's café"` stays readable) by writing to `path + ".tmp"` and then calling `os.replace`. Use `try`/`finally` so the temporary file is removed if anything fails, and so it never remains afterwards. The error itself should still reach the caller.

In [ ]:
import json
import os

def save_data(path, data):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import json as _json, os as _os\nGOOD = {\"expenses\": [{\"id\": 1, \"date\": \"2026-11-02\", \"amount\": 1250, \"category\": \"food\", \"note\": \"Zo\u00eb's caf\u00e9\"}],\n        \"budgets\": {}, \"next_id\": 2}\ndef test_saves():\n    \"saves readable JSON, with nothing left over\"\n    save_data(\"spend.json\", GOOD)\n    text = read_file(\"spend.json\")\n    assert _json.loads(text) == GOOD and \"Zo\u00eb's caf\u00e9\" in text and '\\n  \"expenses\"' in text\n    assert not _os.path.exists(\"spend.json.tmp\")\ndef test_replaces():\n    \"a second save replaces the first\"\n    save_data(\"spend.json\", GOOD)\n    save_data(\"spend.json\", {\"expenses\": [], \"budgets\": {}, \"next_id\": 9})\n    assert _json.loads(read_file(\"spend.json\"))[\"next_id\"] == 9\ndef test_failure_keeps_old():\n    \"a failed save leaves the old file untouched, and no temporary file\"\n    save_data(\"spend.json\", GOOD)\n    before = read_file(\"spend.json\")\n    try:\n        save_data(\"spend.json\", {\"expenses\": [], \"budgets\": {\"bad\": {1, 2}}, \"next_id\": 1})\n        assert False, \"saving a set should raise TypeError\"\n    except TypeError:\n        pass\n    assert read_file(\"spend.json\") == before, \"the old data was damaged\"\n    assert not _os.path.exists(\"spend.json.tmp\"), \"the temporary file was left behind\"")

### Exercise: Step 16: read a statement

Write `read_statement(text)` returning `(rows, problems)`. Read the text with `csv.DictReader`. For each row, with `reader.line_num` as the line number:

1. parse `Date` (stripped) with `%d/%m/%Y`; if that fails, add the problem `line 10: bad date '3O/10/2026'` and skip the row;
2. the stripped `Amount` must fully match `STATEMENT_AMOUNT`, and its second group must pass `parse_amount`; otherwise `line 11: bad amount '-1O.00'`, and skip it;
3. if the first group isn't `"-"`, it's money in: skip it silently;
4. otherwise add `{"date": ISO date, "amount": pence, "description": ...}`, where the description has its runs of whitespace collapsed to single spaces and is stripped.

In [ ]:
import csv
import io
import re
from datetime import datetime
from money import parse_amount

STATEMENT_AMOUNT = re.compile(r"(-?)(£?[\d,]+(?:\.\d{1,2})?)")

def read_statement(text):
    ___

rows, problems = read_statement(open("statement.csv").read())
for r in rows:
    print(r)
print(problems)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_file():\n    \"statement.csv\"\n    rows, problems = read_statement(read_file(\"statement.csv\"))\n    assert problems == [\"line 10: bad date '3O/10/2026'\", \"line 11: bad amount '-1O.00'\"], problems\n    assert rows[0] == {\"date\": \"2026-11-02\", \"amount\": 6210, \"description\": \"TESCO STORES 2041\"}\n    assert [r[\"amount\"] for r in rows] == [6210, 420, 420, 320000, 1099, 380, 645]\n    assert rows[-1][\"description\"] == \"PRET A MANGER\" and rows[3][\"description\"] == \"CITY LETTINGS RENT\"\ndef test_more():\n    \"other cases\"\n    text = \"Date,Description,Amount\\n01/11/2026,REFUND,5.00\\n01/11/2026,X,-0.00\\n1/11/2026,Y,-1\\n01/11/2026,Z,-1.234\\n\"\n    rows, problems = read_statement(text)\n    assert rows == [{\"date\": \"2026-11-01\", \"amount\": 100, \"description\": \"Y\"}]\n    assert problems == [\"line 3: bad amount '-0.00'\", \"line 5: bad amount '-1.234'\"], problems\n    assert read_statement(\"Date,Description,Amount\\n\") == ([], [])")

### Exercise: Step 17: guess a category

Write `guess_category(description, rules=RULES)`: return the category of the first rule whose pattern is found **anywhere** in the description, ignoring case, or `"other"`.

In [ ]:
import re

RULES = [
    (r"tesco|sainsbury|aldi|lidl|co-?op", "food"),
    (r"\btfl\b|trainline|uber|bus", "travel"),
    (r"netflix|spotify|cinema", "fun"),
    (r"rent|lettings", "rent"),
]

def guess_category(description, rules=RULES):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"the default rules\"\n    cases = {\"TESCO STORES 2041\": \"food\", \"Co-op Food\": \"food\", \"COOP\": \"food\", \"TFL TRAVEL CH\": \"travel\",\n             \"NETFLIX.COM\": \"fun\", \"CITY LETTINGS RENT\": \"rent\", \"CORNER CAFE\": \"other\", \"\": \"other\",\n             \"STFLORIAN\": \"other\", \"Uber *Trip\": \"travel\"}\n    for text, want in cases.items():\n        assert guess_category(text) == want, f\"guess_category({text!r}) should be {want!r}\"\ndef test_order():\n    \"the first matching rule wins, and rules can be passed in\"\n    rules = [(r\"cafe\", \"eating out\"), (r\"corner\", \"food\")]\n    assert guess_category(\"CORNER CAFE\", rules) == \"eating out\"\n    assert guess_category(\"Corner shop\", rules) == \"food\"\n    assert guess_category(\"TESCO\", rules) == \"other\"")

### Exercise: Step 18: import a statement

Write `import_statement(data, text, rules=RULES)` that reads the statement with `read_statement`, then for each row:

- the note is the description, cut to 60 characters;
- if a `Counter` of the existing expenses' `(date, amount, note)` still has a count above 0 for this row's key, take one off and count the row as skipped;
- otherwise add an expense directly to `data` (with the next id, and the category from `guess_category`) and count it as added.

Return `{"added": ..., "skipped": ..., "problems": [...]}` with the problems from `read_statement`.

In [ ]:
import csv
import io
import re
from collections import Counter
from datetime import datetime
from core import new_data
from money import parse_amount

STATEMENT_AMOUNT = re.compile(r"(-?)(£?[\d,]+(?:\.\d{1,2})?)")
RULES = [
    (r"tesco|sainsbury|aldi|lidl|co-?op", "food"),
    (r"\btfl\b|trainline|uber|bus", "travel"),
    (r"netflix|spotify|cinema", "fun"),
    (r"rent|lettings", "rent"),
]

def read_statement(text):
    """A bank CSV (Date, Description, Amount) as (rows, problems). Only money out is kept."""
    rows, problems = [], []
    reader = csv.DictReader(io.StringIO(text))
    for row in reader:
        line = reader.line_num
        try:
            when = datetime.strptime(row["Date"].strip(), "%d/%m/%Y").date().isoformat()
        except ValueError:
            problems.append(f"line {line}: bad date {row['Date'].strip()!r}")
            continue
        m = STATEMENT_AMOUNT.fullmatch(row["Amount"].strip())
        try:
            if m is None:
                raise ValueError
            pence = parse_amount(m.group(2))
        except ValueError:
            problems.append(f"line {line}: bad amount {row['Amount'].strip()!r}")
            continue
        if m.group(1) != "-":
            continue
        rows.append({"date": when, "amount": pence, "description": " ".join(row["Description"].split())})
    return rows, problems


def guess_category(description, rules=RULES):
    for pattern, category in rules:
        if re.search(pattern, description, re.IGNORECASE):
            return category
    return "other"


def import_statement(data, text, rules=RULES):
    ___

data = new_data()
text = open("statement.csv").read()
print(import_statement(data, text))
print(import_statement(data, text))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from core import new_data as _new\ndef test_first_and_again():\n    \"importing twice adds nothing the second time\"\n    d = _new()\n    text = read_file(\"statement.csv\")\n    r = import_statement(d, text)\n    assert r == {\"added\": 7, \"skipped\": 0, \"problems\": [\"line 10: bad date '3O/10/2026'\", \"line 11: bad amount '-1O.00'\"]}, r\n    assert [e[\"id\"] for e in d[\"expenses\"]] == [1, 2, 3, 4, 5, 6, 7] and d[\"next_id\"] == 8\n    assert d[\"expenses\"][0] == {\"id\": 1, \"date\": \"2026-11-02\", \"amount\": 6210, \"category\": \"food\", \"note\": \"TESCO STORES 2041\"}\n    assert [e[\"category\"] for e in d[\"expenses\"]] == [\"food\", \"travel\", \"travel\", \"rent\", \"fun\", \"other\", \"other\"]\n    r2 = import_statement(d, text)\n    assert r2[\"added\"] == 0 and r2[\"skipped\"] == 7 and len(d[\"expenses\"]) == 7\ndef test_counts():\n    \"duplicates are counted, not just spotted\"\n    d = _new()\n    d[\"expenses\"].append({\"id\": 1, \"date\": \"2026-11-02\", \"amount\": 420, \"category\": \"travel\", \"note\": \"TFL TRAVEL CH\"})\n    d[\"next_id\"] = 5\n    text = \"Date,Description,Amount\\n02/11/2026,TFL TRAVEL CH,-4.20\\n02/11/2026,TFL   TRAVEL CH,-4.20\\n\"\n    r = import_statement(d, text)\n    assert r[\"added\"] == 1 and r[\"skipped\"] == 1 and d[\"expenses\"][-1][\"id\"] == 5 and d[\"next_id\"] == 6\ndef test_long_and_rules():\n    \"long notes are cut, and rules can be passed in\"\n    d = _new()\n    long = \"A\" * 70\n    text = f\"Date,Description,Amount\\n01/11/2026,{long},-1.00\\n\"\n    import_statement(d, text, rules=[(\"a\", \"letters\")])\n    assert d[\"expenses\"][0][\"note\"] == \"A\" * 60 and d[\"expenses\"][0][\"category\"] == \"letters\"\n    assert import_statement(d, text)[\"skipped\"] == 1")

### Exercise: Step 19: export CSV

Write `export_csv(data, path)` that writes a CSV with the header `id,date,amount,category,note`, and one row per expense, oldest first (by date, then id). Amounts are written as plain pounds with two decimals and no `£` or commas: `320000` as `3200.00`, so a spreadsheet reads them as numbers.

In [ ]:
import csv

def export_csv(data, path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("D = {\"expenses\": [\n    {\"id\": 3, \"date\": \"2026-11-02\", \"amount\": 320000, \"category\": \"rent\", \"note\": \"\"},\n    {\"id\": 1, \"date\": \"2026-11-02\", \"amount\": 5, \"category\": \"food\", \"note\": 'Tola\\'s \"big\" lunch, with cake'},\n    {\"id\": 2, \"date\": \"2026-10-30\", \"amount\": 1250, \"category\": \"eating out\", \"note\": \"Zo\u00eb\"},\n], \"budgets\": {}, \"next_id\": 4}\ndef test_export():\n    \"the file\"\n    export_csv(D, \"out.csv\")\n    want = ('id,date,amount,category,note\\r\\n2,2026-10-30,12.50,eating out,Zo\u00eb\\r\\n'\n            '1,2026-11-02,0.05,food,\"Tola\\'s \"\"big\"\" lunch, with cake\"\\r\\n3,2026-11-02,3200.00,rent,\\r\\n')\n    assert read_file(\"out.csv\") == want, read_file(\"out.csv\")\ndef test_read_back():\n    \"the csv module reads it back\"\n    import csv as _csv\n    export_csv(D, \"out.csv\")\n    with open(\"out.csv\", newline=\"\", encoding=\"utf-8\") as f:\n        rows = list(_csv.DictReader(f))\n    assert rows[1][\"note\"] == 'Tola\\'s \"big\" lunch, with cake' and rows[2][\"amount\"] == \"3200.00\"")

### Exercise: Step 20: test the storage

Write tests for `storage.py`, using `tmp_path` (and a fixture if it helps). They must pass on the real code and catch all **eight** broken versions. Paths in `storage.py` are plain strings, so pass `str(tmp_path / "spend.json")`.

In [ ]:
# Run this cell first: it creates the files used below.
with open('storage.py', 'w', encoding='utf-8') as f:
    f.write('"""Saving and loading the data, and importing and exporting CSV."""\nimport csv\nimport io\nimport json\nimport os\nimport re\nfrom collections import Counter\nfrom datetime import datetime\n\nfrom core import new_data\nfrom money import parse_amount\n\nKEYS = ["id", "date", "amount", "category", "note"]\n\n\ndef load_data(path):\n    """The data saved at path, or new_data() if there\'s no file. ValueError if it\'s damaged."""\n    if not os.path.exists(path):\n        return new_data()\n    with open(path, encoding="utf-8") as f:\n        try:\n            data = json.load(f)\n        except json.JSONDecodeError as e:\n            raise ValueError(f"{path} is damaged: {e.msg} at line {e.lineno}") from None\n    if not (isinstance(data, dict) and isinstance(data.get("expenses"), list)\n            and isinstance(data.get("budgets"), dict) and isinstance(data.get("next_id"), int)):\n        raise ValueError(f"{path} is not a spend data file")\n    for e in data["expenses"]:\n        if not (isinstance(e, dict) and list(e) == KEYS and isinstance(e["amount"], int)):\n            raise ValueError(f"{path} has a damaged expense: {e!r}")\n    return data\n\n\ndef save_data(path, data):\n    """Save safely: write a temporary file, then replace the old file with it in one step."""\n    tmp = path + ".tmp"\n    try:\n        with open(tmp, "w", encoding="utf-8") as f:\n            json.dump(data, f, indent=2, ensure_ascii=False)\n        os.replace(tmp, path)\n    finally:\n        if os.path.exists(tmp):\n            os.remove(tmp)\n\n\nSTATEMENT_AMOUNT = re.compile(r"(-?)(£?[\\d,]+(?:\\.\\d{1,2})?)")\n\n\ndef read_statement(text):\n    """A bank CSV (Date, Description, Amount) as (rows, problems). Only money out is kept."""\n    rows, problems = [], []\n    reader = csv.DictReader(io.StringIO(text))\n    for row in reader:\n        line = reader.line_num\n        try:\n            when = datetime.strptime(row["Date"].strip(), "%d/%m/%Y").date().isoformat()\n        except ValueError:\n            problems.append(f"line {line}: bad date {row[\'Date\'].strip()!r}")\n            continue\n        m = STATEMENT_AMOUNT.fullmatch(row["Amount"].strip())\n        try:\n            if m is None:\n                raise ValueError\n            pence = parse_amount(m.group(2))\n        except ValueError:\n            problems.append(f"line {line}: bad amount {row[\'Amount\'].strip()!r}")\n            continue\n        if m.group(1) != "-":\n            continue\n        rows.append({"date": when, "amount": pence, "description": " ".join(row["Description"].split())})\n    return rows, problems\n\n\nRULES = [\n    (r"tesco|sainsbury|aldi|lidl|co-?op", "food"),\n    (r"\\btfl\\b|trainline|uber|bus", "travel"),\n    (r"netflix|spotify|cinema", "fun"),\n    (r"rent|lettings", "rent"),\n]\n\n\ndef guess_category(description, rules=RULES):\n    for pattern, category in rules:\n        if re.search(pattern, description, re.IGNORECASE):\n            return category\n    return "other"\n\n\ndef import_statement(data, text, rules=RULES):\n    rows, problems = read_statement(text)\n    existing = Counter((e["date"], e["amount"], e["note"]) for e in data["expenses"])\n    added = skipped = 0\n    for r in rows:\n        note = r["description"][:60]\n        key = (r["date"], r["amount"], note)\n        if existing[key] > 0:\n            existing[key] -= 1\n            skipped += 1\n            continue\n        data["expenses"].append({"id": data["next_id"], "date": r["date"], "amount": r["amount"],\n                                 "category": guess_category(r["description"], rules), "note": note})\n        data["next_id"] += 1\n        added += 1\n    return {"added": added, "skipped": skipped, "problems": problems}\n\n\ndef export_csv(data, path):\n    with open(path, "w", newline="", encoding="utf-8") as f:\n        w = csv.writer(f)\n        w.writerow(["id", "date", "amount", "category", "note"])\n        for e in sorted(data["expenses"], key=lambda e: (e["date"], e["id"])):\n            w.writerow([e["id"], e["date"], f"{e[\'amount\'] // 100}.{e[\'amount\'] % 100:02d}", e["category"], e["note"]])\n')

In [ ]:
import json
import os
import pytest
from core import new_data
from storage import load_data, save_data, import_statement, export_csv

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"storage.py\").read()\nBROKEN = {\n    \"missing file crashes\": GOOD.replace('    if not os.path.exists(path):\\n        return new_data()\\n', \"\"),\n    \"damaged file treated as empty\": GOOD.replace(\n        '            raise ValueError(f\"{path} is damaged: {e.msg} at line {e.lineno}\") from None', \"            return new_data()\"),\n    \"no structure check\": GOOD.replace('        raise ValueError(f\"{path} is not a spend data file\")', \"        pass\"),\n    \"saves in place\": GOOD.replace('''    tmp = path + \".tmp\"\n    try:\n        with open(tmp, \"w\", encoding=\"utf-8\") as f:\n            json.dump(data, f, indent=2, ensure_ascii=False)\n        os.replace(tmp, path)\n    finally:\n        if os.path.exists(tmp):\n            os.remove(tmp)''', '''    with open(path, \"w\", encoding=\"utf-8\") as f:\n        json.dump(data, f, indent=2, ensure_ascii=False)'''),\n    \"leaves the temporary file\": GOOD.replace('''    finally:\n        if os.path.exists(tmp):\n            os.remove(tmp)''', '''    except Exception:\n        raise'''),\n    \"money in imported\": GOOD.replace('        if m.group(1) != \"-\":\\n            continue\\n', \"\"),\n    \"duplicates only spotted\": GOOD.replace(\"            existing[key] -= 1\\n\", \"\"),\n    \"export newest first\": GOOD.replace('key=lambda e: (e[\"date\"], e[\"id\"]))', 'key=lambda e: (e[\"date\"], e[\"id\"]), reverse=True)'),\n}\ndef test_style():\n    \"uses tmp_path\"\n    assert \"tmp_path\" in source\ndef test_catches():\n    \"your tests pass on the real code and catch all eight broken versions\"\n    results = against(source, \"storage\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"storage.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the real code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

> **Files are the edge.** Everything that can go wrong with spend's data goes wrong at the file: a crash mid-save, a damaged file, a statement with typos. `storage.py` is small, but it's where the care went, and it's the part with the most tests. Notice also what isn't here: no printing, and no decisions about what to tell the person. Those belong to the interface, which you'll build tomorrow.

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Where we are

The modules from Days 1 and 2 are in the files below. `dates.py` now also holds `parse_month` and `next_month`, and `core.py` has everything from yesterday, including `monthly_totals` and the budget functions. Today's work is a new module, `storage.py`: the only part of spend that touches files.

There's also `statement.csv`, an export from a bank. Open it now and look for the rows that will cause trouble. (There are two broken rows, a payment in, and two rows that look the same.)

## Step 14: loading

When spend starts, it loads the data file. Three things can be wrong, and each needs a different response:

- **No file yet** is normal: it's the first time spend has run. Start with empty data.
- **A damaged file**, one that isn't valid JSON (half-written, or edited by hand and broken), must **not** be treated as "no data". If spend started empty and then saved, it would overwrite the file and lose everything in it. Stop, and say what's wrong.
- **The wrong file**, valid JSON but not spend's (someone pointed it at a different file), likewise: stop with a message.

This is Week 23's lesson again: check data at the edge, as it comes in, so the rest of the program can trust it.

## Step 15: saving safely

The obvious way to save is to open the file for writing and dump the JSON into it. But opening a file with `"w"` **empties it at once**. If anything goes wrong while writing (the program crashes, the disk fills up, the battery dies, or the data contains something JSON can't store), the old data is already gone and the new data is only half there.

The safe way, used by editors, databases and nearly every program that saves files:

1. write the new data to a **temporary file** next to the real one (`spend.json.tmp`);
2. only when that has finished, **replace** the real file with it using `os.replace`, which swaps them in one step: anyone looking sees either the whole old file or the whole new one, never a mix;
3. if anything fails, delete the temporary file and leave the real one alone.

This is called an **atomic** write: it happens completely or not at all, like `add_expense`.

Why not catch the error and print a message here? Because `storage.py` doesn't print: it lets the error reach the interface, which decides what to tell the person. A library function that swallowed errors would leave its caller believing the save worked.

## Step 16: reading a bank statement

Typing in every coffee is tedious, and most banks let you download a statement as CSV. Each bank's format is a little different; this one, like many, has a date (day first), a description, and a signed amount: **negative** for money out, positive for money in.

spend tracks spending, so only money out becomes an expense, as a positive number of pence. Money in (salary, refunds) is skipped without fuss. Rows that can't be read are reported with their line number, like the gradebook in Week 12.

## Step 17: guessing categories

Bank descriptions are written for the bank, not for you: `TESCO STORES 2041`, `TFL TRAVEL CH`. A list of **rules**, each a regular expression and a category, turns most of them into something useful. The first rule that matches wins, and anything unmatched becomes `"other"`, for the person to fix later. A list (not a dictionary) because the order matters: put specific rules before general ones.

## Step 18: importing without duplicates

People download a statement, import it, and next week download another that overlaps the first. Importing must not add the same payment twice. An imported expense keeps the bank's description as its note, so a row is already imported if there's an expense with the same date, amount and note.

But look at `statement.csv` again: two bus fares, same day, same amount, same description. They're **two real payments**. If you only asked "is there already one like this?", the second fare would be skipped the first time you imported. The fix is to **count**: if there's one matching expense and the statement has two rows, one is new. A `Counter` (Week 20) keeps the counts.

## Step 19: exporting

The last piece: a CSV a spreadsheet can open, oldest first. The `csv` module handles notes containing commas or quotes, which is exactly why you use it instead of joining strings with commas (Week 23).

## Step 20: testing the storage

Here's `storage.py` complete. File code is where tests most need care: they must never touch your real data, and they must check what happens when things go wrong, not just when they go right. `tmp_path` gives every test its own empty folder.

## Quiz

## Before you go

Add `storage.py` and `tests/test_storage.py` to your `spend` folder. Run all the tests (`pytest`), and commit when they're green. Tomorrow: the command-line interface that ties everything together, so you can finally type `python3 spend.py add 12.50 food`.